In [3]:
!pip install yfinance --quiet

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

print("TF version:", tf.__version__)

# =========================================
# 1. CONFIG
# =========================================

TICKERS = [
    "RELIANCE.NS",
    "TCS.NS",
    "HDFCBANK.NS",
    "INFY.NS",
]

START_DATE = "2013-01-01"
END_DATE   = "2024-12-31"

LOOKBACK   = 60      # window length (days)
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15
RISK_FRAC  = 0.10    # top 10% = big moves / risk flags

# =========================================
# 2. DOWNLOAD + FEATURE ENGINEERING PER STOCK
# =========================================

def add_features(df):
    df = df.copy()
    # Basic returns
    df["log_ret"] = np.log(df["Close"] / df["Close"].shift(1))

    # Simple moving averages
    df["sma_5"]  = df["Close"].rolling(5).mean()
    df["sma_10"] = df["Close"].rolling(10).mean()
    df["sma_20"] = df["Close"].rolling(20).mean()

    # Volatility (std of log returns)
    df["vol_5"]  = df["log_ret"].rolling(5).std()
    df["vol_10"] = df["log_ret"].rolling(10).std()

    # RSI 14
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    df = df.dropna().copy()
    return df

feature_cols = [
    "Open", "High", "Low", "Close", "Volume",
    "log_ret",
    "sma_5", "sma_10", "sma_20",
    "vol_5", "vol_10",
    "rsi_14",
]

raw_data = {}
feat_data = {}

for ticker in TICKERS:
    print(f"\nDownloading {ticker}...")
    df_raw = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True)
    print("  Raw shape:", df_raw.shape)
    df_raw = df_raw[["Open", "High", "Low", "Close", "Volume"]].copy()
    df_feat = add_features(df_raw)
    df_feat["Ticker"] = ticker
    raw_data[ticker] = df_raw
    feat_data[ticker] = df_feat
    print("  With features shape:", df_feat.shape)

# =========================================
# 3. FIT GLOBAL SCALER ON ALL STOCKS
# =========================================

all_feat = np.vstack([feat_data[t][feature_cols].values for t in TICKERS])
scaler = StandardScaler()
scaler.fit(all_feat)

close_index = feature_cols.index("Close")
price_mean = float(scaler.mean_[close_index])
price_std  = float(scaler.scale_[close_index])

print("\nScaler Close mean:", price_mean)
print("Scaler Close std :", price_std)

# =========================================
# 4. CREATE WINDOWS PER STOCK, THEN SPLIT BY TIME PER STOCK
# =========================================

ticker_to_id = {t: i for i, t in enumerate(TICKERS)}
num_stocks = len(TICKERS)

X_train_list, y_train_list = [], []
X_val_list,   y_val_list   = [], []
X_test_list,  y_test_list  = [], []

prev_train_list, prev_val_list, prev_test_list = [], [], []
true_train_list, true_val_list, true_test_list = [], [], []
dates_train_list, dates_val_list, dates_test_list = [], [], []

stockid_train_list, stockid_val_list, stockid_test_list = [], [], []

r_train_list, r_val_list, r_test_list = [], [], []  # returns for classification labels

for ticker in TICKERS:
    df = feat_data[ticker].copy()
    df = df.sort_index()
    feat_scaled = scaler.transform(df[feature_cols].values)

    close_prices = df["Close"].values
    dates = df.index.values

    X_t, y_t = [], []
    prev_t, true_t, dates_t, r_t = [], [], [], []

    for i in range(LOOKBACK, len(df)):
        window = feat_scaled[i-LOOKBACK:i, :]      # (60, F)
        y_norm = feat_scaled[i, close_index]       # normalized Close
        P_prev = close_prices[i-1]
        P_next = close_prices[i]
        ret    = (P_next - P_prev) / P_prev

        X_t.append(window)
        y_t.append(y_norm)
        prev_t.append(P_prev)
        true_t.append(P_next)
        dates_t.append(dates[i])
        r_t.append(ret)

    X_t = np.array(X_t, dtype=np.float32)
    y_t = np.array(y_t, dtype=np.float32)
    prev_t = np.array(prev_t, dtype=np.float32)
    true_t = np.array(true_t, dtype=np.float32)
    dates_t = np.array(dates_t)
    r_t = np.array(r_t, dtype=np.float32)

    N_t = len(X_t)
    n_test_t = int(N_t * TEST_FRAC)
    n_val_t  = int(N_t * VAL_FRAC)
    n_train_t = N_t - n_val_t - n_test_t

    sid = ticker_to_id[ticker]

    # Train
    X_train_list.append(X_t[:n_train_t])
    y_train_list.append(y_t[:n_train_t])
    prev_train_list.append(prev_t[:n_train_t])
    true_train_list.append(true_t[:n_train_t])
    dates_train_list.append(dates_t[:n_train_t])
    stockid_train_list.append(np.full(n_train_t, sid, dtype=int))
    r_train_list.append(r_t[:n_train_t])

    # Val
    X_val_list.append(X_t[n_train_t:n_train_t + n_val_t])
    y_val_list.append(y_t[n_train_t:n_train_t + n_val_t])
    prev_val_list.append(prev_t[n_train_t:n_train_t + n_val_t])
    true_val_list.append(true_t[n_train_t:n_train_t + n_val_t])
    dates_val_list.append(dates_t[n_train_t:n_train_t + n_val_t])
    stockid_val_list.append(np.full(n_val_t, sid, dtype=int))
    r_val_list.append(r_t[n_train_t:n_train_t + n_val_t])

    # Test
    X_test_list.append(X_t[n_train_t + n_val_t:])
    y_test_list.append(y_t[n_train_t + n_val_t:])
    prev_test_list.append(prev_t[n_train_t + n_val_t:])
    true_test_list.append(true_t[n_train_t + n_val_t:])
    dates_test_list.append(dates_t[n_train_t + n_val_t:])
    stockid_test_list.append(np.full(n_test_t, sid, dtype=int))
    r_test_list.append(r_t[n_train_t + n_val_t:])

# Concatenate across stocks
def cat(lst, axis=0):
    return np.concatenate(lst, axis=axis) if len(lst) > 1 else lst[0]

X_train = cat(X_train_list)
y_train = cat(y_train_list)
prev_train = cat(prev_train_list)
true_train = cat(true_train_list)
dates_train = cat(dates_train_list)
stockid_train = cat(stockid_train_list)
r_train = cat(r_train_list)

X_val = cat(X_val_list)
y_val = cat(y_val_list)
prev_val = cat(prev_val_list)
true_val = cat(true_val_list)
dates_val = cat(dates_val_list)
stockid_val = cat(stockid_val_list)
r_val = cat(r_val_list)

X_test = cat(X_test_list)
y_test = cat(y_test_list)
prev_test = cat(prev_test_list)
true_test = cat(true_test_list)
dates_test = cat(dates_test_list)
stockid_test = cat(stockid_test_list)
r_test = cat(r_test_list)

print("\nShapes after concat:")
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

# =========================================
# 5. ADD STOCK ONE-HOT FEATURES TO EACH TIMESTEP
# =========================================

def add_stock_one_hot(X, stock_ids, num_stocks):
    """
    X: (N, T, F)
    stock_ids: (N,)
    Returns: X_ext = (N, T, F + num_stocks)
    """
    X = np.asarray(X)
    stock_ids = np.asarray(stock_ids).reshape(-1)
    N, T, F = X.shape
    one_hot = np.eye(num_stocks)[stock_ids]        # (N, num_stocks)
    one_hot_rep = np.repeat(one_hot[:, None, :], T, axis=1)  # (N, T, num_stocks)
    X_ext = np.concatenate([X, one_hot_rep], axis=-1)
    return X_ext

X_train = add_stock_one_hot(X_train, stockid_train, num_stocks)
X_val   = add_stock_one_hot(X_val,   stockid_val,   num_stocks)
X_test  = add_stock_one_hot(X_test,  stockid_test,  num_stocks)

print("After stock one-hot:")
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

# =========================================
# 6. CREATE BIG-MOVE LABELS FOR CLASSIFICATION HEAD
#    (based on TRAIN returns; same threshold used on val/test)
# =========================================

abs_r_train = np.abs(r_train)
big_thr_train = np.quantile(abs_r_train, 1 - RISK_FRAC)  # top 10% of train moves
print("\nTrain big-move threshold |r| >= ", big_thr_train)

def big_move_labels(r, thr):
    r = np.asarray(r).reshape(-1)
    return (np.abs(r) >= thr).astype(np.float32)

y_big_train = big_move_labels(r_train, big_thr_train)
y_big_val   = big_move_labels(r_val,   big_thr_train)
y_big_test  = big_move_labels(r_test,  big_thr_train)

print("Big-move counts -> train:", int(y_big_train.sum()),
      "val:", int(y_big_val.sum()),
      "test:", int(y_big_test.sum()))

# Keras likes shape (N,1) for regression and binary labels
y_train_reg = y_train.reshape(-1, 1)
y_val_reg   = y_val.reshape(-1, 1)
y_test_reg  = y_test.reshape(-1, 1)

y_train_cls = y_big_train.reshape(-1, 1)
y_val_cls   = y_big_val.reshape(-1, 1)
y_test_cls  = y_big_test.reshape(-1, 1)


[*********************100%***********************]  1 of 1 completed

TF version: 2.19.0

  Raw shape: (2958, 5)
  With features shape: (2939, 13)




[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


  Raw shape: (2958, 5)
  With features shape: (2939, 13)

  Raw shape: (2958, 5)


[*********************100%***********************]  1 of 1 completed

  With features shape: (2939, 13)



  Raw shape: (2958, 5)
  With features shape: (2939, 13)

Scaler Close mean: 954.4847830798035
Scaler Close std : 842.0290740244528

Shapes after concat:
X_train: (8068, 60, 12) X_val: (1724, 60, 12) X_test: (1724, 60, 12)
After stock one-hot:
X_train: (8068, 60, 16) X_val: (1724, 60, 16) X_test: (1724, 60, 16)

Train big-move threshold |r| >=  0.024816869
Big-move counts -> train: 807 val: 150 test: 97


In [4]:
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers

print("TF version:", tf.__version__)

# =========================================
# 1. TEMPORAL ATTENTION LAYER
# =========================================

class TemporalAttention(layers.Layer):
    """
    Simple additive temporal attention over a sequence of hidden states.
    Input:  (batch, T, H)
    Output: context vector (batch, H)
    """
    def __init__(self, attn_units=32, **kwargs):
        super().__init__(**kwargs)
        self.attn_units = attn_units

    def build(self, input_shape):
        H = int(input_shape[-1])
        self.W_h = self.add_weight(
            shape=(H, self.attn_units),
            initializer="glorot_uniform",
            name="W_h")
        self.b_h = self.add_weight(
            shape=(self.attn_units,),
            initializer="zeros",
            name="b_h")
        self.v = self.add_weight(
            shape=(self.attn_units, 1),
            initializer="glorot_uniform",
            name="v")
        super().build(input_shape)

    def call(self, inputs):
        # inputs: (B, T, H)
        score = tf.tensordot(inputs, self.W_h, axes=[[2], [0]])  # (B, T, attn_units)
        score = tf.tanh(score + self.b_h)
        score = tf.tensordot(score, self.v, axes=[[2], [0]])     # (B, T, 1)
        score = tf.squeeze(score, axis=-1)                       # (B, T)
        alpha = tf.nn.softmax(score, axis=-1)                    # (B, T)
        alpha_expanded = tf.expand_dims(alpha, axis=-1)          # (B, T, 1)
        context = tf.reduce_sum(alpha_expanded * inputs, axis=1) # (B, H)
        return context

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"attn_units": self.attn_units})
        return cfg

# =========================================
# 2. MODEL DEFINITIONS
# =========================================

REG_L2 = 1e-4

def build_cnn_bilstm_baseline(input_shape):
    inp = layers.Input(shape=input_shape, name="input_series")

    x = layers.Conv1D(
        filters=32,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(REG_L2),
        name="conv1d_base"
    )(inp)
    x = layers.BatchNormalization(name="bn_base")(x)
    x = layers.Dropout(0.25, name="dropout_base")(x)

    x = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(REG_L2),
            recurrent_regularizer=regularizers.l2(REG_L2),
            name="bilstm1_inner"
        ),
        name="bilstm1"
    )(x)
    x = layers.Dropout(0.3, name="dropout_bilstm1")(x)

    x = layers.Bidirectional(
        layers.LSTM(
            32,
            return_sequences=False,
            kernel_regularizer=regularizers.l2(REG_L2),
            recurrent_regularizer=regularizers.l2(REG_L2),
            name="bilstm2_inner"
        ),
        name="bilstm2"
    )(x)
    x = layers.Dropout(0.3, name="dropout_bilstm2")(x)

    price_out = layers.Dense(
        1,
        kernel_regularizer=regularizers.l2(REG_L2),
        name="price_out"
    )(x)

    model = models.Model(inputs=inp, outputs=price_out, name="CNN_BiLSTM_Baseline")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=["mae", "mse"]
    )
    return model

def build_msf_cal_multitask(input_shape):
    inp = layers.Input(shape=input_shape, name="input_series")

    # Multi-scale convs
    conv3 = layers.Conv1D(
        24, kernel_size=3, padding="same", activation="relu",
        kernel_regularizer=regularizers.l2(REG_L2),
        name="conv_k3"
    )(inp)
    conv5 = layers.Conv1D(
        24, kernel_size=5, padding="same", activation="relu",
        kernel_regularizer=regularizers.l2(REG_L2),
        name="conv_k5"
    )(inp)
    conv7 = layers.Conv1D(
        24, kernel_size=7, padding="same", activation="relu",
        kernel_regularizer=regularizers.l2(REG_L2),
        name="conv_k7"
    )(inp)

    x = layers.Concatenate(name="concat_multiscale")([conv3, conv5, conv7])
    x = layers.BatchNormalization(name="bn_multiscale")(x)
    x = layers.Dropout(0.25, name="dropout_multiscale")(x)

    x = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(REG_L2),
            recurrent_regularizer=regularizers.l2(REG_L2),
            name="bilstm1_inner"
        ),
        name="bilstm1"
    )(x)
    x = layers.Dropout(0.3, name="dropout_bilstm1")(x)

    x = layers.Bidirectional(
        layers.LSTM(
            32,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(REG_L2),
            recurrent_regularizer=regularizers.l2(REG_L2),
            name="bilstm2_inner"
        ),
        name="bilstm2"
    )(x)
    x = layers.Dropout(0.3, name="dropout_bilstm2")(x)

    context = TemporalAttention(attn_units=48, name="temporal_attention")(x)

    # Shared dense
    h = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(REG_L2),
        name="dense_shared"
    )(context)
    h = layers.Dropout(0.3, name="dropout_shared")(h)

    # Head 1: price regression (normalized Close)
    price_out = layers.Dense(
        1,
        kernel_regularizer=regularizers.l2(REG_L2),
        name="price_out"
    )(h)

    # Head 2: big-move probability
    risk_out = layers.Dense(
        1,
        activation="sigmoid",
        kernel_regularizer=regularizers.l2(REG_L2),
        name="risk_out"
    )(h)

    model = models.Model(inputs=inp, outputs=[price_out, risk_out], name="MSF_CAL_MultiTask")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss={
            "price_out": tf.keras.losses.Huber(delta=1.0),
            "risk_out": "binary_crossentropy",
        },
        loss_weights={"price_out": 0.5, "risk_out": 0.5},
        metrics={
            "price_out": ["mae", "mse"],
            "risk_out": ["accuracy"],
        }
    )
    return model

input_shape = X_train.shape[1:]
print("Input shape:", input_shape)

baseline = build_cnn_bilstm_baseline(input_shape)
msf_cal  = build_msf_cal_multitask(input_shape)

baseline.summary()
msf_cal.summary()

# =========================================
# 3. TRAINING
# =========================================

EPOCHS = 60
BATCH_SIZE = 32

es = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)
rlr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("\n================ TRAINING BASELINE (regression only) ================\n")
hist_base = baseline.fit(
    X_train, y_train_reg,
    validation_data=(X_val, y_val_reg),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es, rlr],
    verbose=1
)

print("\n================ TRAINING MSF-CAL (multi-task) ================\n")
hist_msf = msf_cal.fit(
    X_train,
    {"price_out": y_train_reg, "risk_out": y_train_cls},
    validation_data=(X_val, {"price_out": y_val_reg, "risk_out": y_val_cls}),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es, rlr],
    verbose=1
)

print("\n=== Test evaluation (normalized price) ===")
# Baseline
loss_b, mae_b, mse_b = baseline.evaluate(X_test, y_test_reg, verbose=0)
print(f"Baseline -> loss: {loss_b:.4f}, MAE: {mae_b:.4f}, MSE: {mse_b:.4f}")

# MSF-CAL
eval_msf = msf_cal.evaluate(
    X_test,
    {"price_out": y_test_reg, "risk_out": y_test_cls},
    verbose=0
)
# eval_msf is a list: [total_loss, price_loss, risk_loss, price_mae, price_mse, risk_acc]
total_loss_m, price_loss_m, risk_loss_m, price_mae_m, price_mse_m, risk_acc_m = eval_msf
print(f"MSF-CAL -> total_loss: {total_loss_m:.4f}, price_MAE: {price_mae_m:.4f}, price_MSE: {price_mse_m:.4f}, risk_acc: {risk_acc_m:.4f}")

# =========================================
# 4. PRICE METRICS IN ₹
# =========================================

def denorm_price(y_norm):
    return y_norm * price_std + price_mean

# Baseline predictions
y_base_norm = baseline.predict(X_test, verbose=0).reshape(-1)
y_msf_price_norm, y_msf_risk_prob = msf_cal.predict(X_test, verbose=0)
y_msf_norm = y_msf_price_norm.reshape(-1)
y_risk_prob = y_msf_risk_prob.reshape(-1)

y_true_norm = y_test_reg.reshape(-1)

y_true_price = denorm_price(y_true_norm)
y_base_price = denorm_price(y_base_norm)
y_msf_price  = denorm_price(y_msf_norm)

def price_metrics(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{name:12s} -> MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}, R^2: {r2:.4f}")

print("\n================ PRICE METRICS (₹) ==================")
price_metrics("Baseline", y_true_price, y_base_price)
price_metrics("MSF-CAL",  y_true_price, y_msf_price)

# =========================================
# 5. RISK-FLAGGING EVALUATION
# =========================================

def compute_returns(next_price, prev_price):
    next_price = np.asarray(next_price).reshape(-1)
    prev_price = np.asarray(prev_price).reshape(-1)
    return (next_price - prev_price) / prev_price

r_true_test = compute_returns(true_test, prev_test)

abs_true = np.abs(r_true_test)
big_move_thr_test = np.quantile(abs_true, 1 - RISK_FRAC)  # top 10% of TEST

y_big_test_true = (abs_true >= big_move_thr_test).astype(int)

print("\nTest big-move threshold |r| >= ", big_move_thr_test)
print("Total test days:", len(r_true_test),
      "Big-move days:", int(y_big_test_true.sum()))

def top_k_flags(risk_scores, frac=RISK_FRAC):
    risk_scores = np.asarray(risk_scores).reshape(-1)
    k = int(len(risk_scores) * frac)
    idx_sorted = np.argsort(-risk_scores)  # descending
    flags = np.zeros_like(risk_scores, dtype=int)
    flags[idx_sorted[:k]] = 1
    return flags

def precision_recall(flags, y_big):
    flags = np.asarray(flags).reshape(-1)
    y_big = np.asarray(y_big).reshape(-1)
    tp = np.sum((flags == 1) & (y_big == 1))
    fp = np.sum((flags == 1) & (y_big == 0))
    fn = np.sum((flags == 0) & (y_big == 1))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    return prec, rec, int(tp)

# Risk scores:
# - Baseline: |predicted return| from price predictions
r_base_pred = compute_returns(y_base_price, prev_test)
risk_base   = np.abs(r_base_pred)

# - MSF-CAL multi-task: use risk_out probability
risk_msf = y_risk_prob  # already in [0,1]

# - Random baseline
np.random.seed(42)
risk_rand = np.random.rand(len(r_true_test))

flags_rand  = top_k_flags(risk_rand)
flags_base  = top_k_flags(risk_base)
flags_msf   = top_k_flags(risk_msf)

print("Flags count -> random:", flags_rand.sum(),
      "baseline:", flags_base.sum(),
      "msf_cal:", flags_msf.sum())

p_rand, r_rand, tp_rand = precision_recall(flags_rand,  y_big_test_true)
p_base, r_base, tp_base = precision_recall(flags_base,  y_big_test_true)
p_msf,  r_msf,  tp_msf  = precision_recall(flags_msf,   y_big_test_true)

print("\n================ RISK FLAGGING (top 10% big moves, TEST) ================")
print(f"Random    -> precision: {p_rand:.3f}, recall: {r_rand:.3f}, TP: {tp_rand}")
print(f"Baseline  -> precision: {p_base:.3f}, recall: {r_base:.3f}, TP: {tp_base}")
print(f"MSF-CAL   -> precision: {p_msf:.3f}, recall: {r_msf:.3f}, TP: {tp_msf}")


TF version: 2.19.0
Input shape: (60, 16)


Model: "CNN_BiLSTM_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_series (InputLayer)       │ (None, 60, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_base (Conv1D)            │ (None, 60, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_base (BatchNormalization)    │ (None, 60, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_base (Dropout)          │ (None, 60, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm1 (Bidirectional)         │ (None, 60, 128)        │        49,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_bilstm1 (Dropout)       │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm2 (Bidirectional)         │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_bilstm2 (Dropout)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ price_out (Dense)               │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,641 (361.88 KB)

 Trainable params: 92,577 (361.63 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_series        │ (None, 60, 16)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k3 (Conv1D)    │ (None, 60, 24)    │      1,176 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k5 (Conv1D)    │ (None, 60, 24)    │      1,944 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k7 (Conv1D)    │ (None, 60, 24)    │      2,712 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_multiscale   │ (None, 60, 72)    │          0 │ conv_k3[0][0],    │
│ (Concatenate)       │                   │            │ conv_k5[0][0],    │
│                     │                   │            │ conv_k7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_multiscale       │ (None, 60, 72)    │        288 │ concat_multiscal… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_multiscale  │ (None, 60, 72)    │          0 │ bn_multiscale[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm1             │ (None, 60, 128)   │     70,144 │ dropout_multisca… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_bilstm1     │ (None, 60, 128)   │          0 │ bilstm1[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm2             │ (None, 60, 64)    │     41,216 │ dropout_bilstm1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_bilstm2     │ (None, 60, 64)    │          0 │ bilstm2[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention  │ (None, 64)        │      3,168 │ dropout_bilstm2[… │
│ (TemporalAttention) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_shared        │ (None, 64)        │      4,160 │ temporal_attenti… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_shared      │ (None, 64)        │          0 │ dense_shared[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_out (Dense)   │ (None, 1)         │         65 │ dropout_shared[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ risk_out (Dense)    │ (None, 1)         │         65 │ dropout_shared[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 124,938 (488.04 KB)

 Trainable params: 124,794 (487.48 KB)

 Non-trainable params: 144 (576.00 B)


================ TRAINING BASELINE (regression only) ================

Epoch 1/60
253/253 ━━━━━━━━━━━━━━━━━━━━ 69s 129ms/step - loss: 0.0911 - mae: 0.1960 - mse: 0.0811 - val_loss: 0.0527 - val_mae: 0.1346 - val_mse: 0.0365 - learning_rate: 0.0010
Epoch 2/60
253/253 ━━━━━━━━━━━━━━━━━━━━ 31s 121ms/step - loss: 0.0411 - mae: 0.1057 - mse: 0.0205 - val_loss: 0.0363 - val_mae: 0.1164 - val_mse: 0.0276 - learning_rate: 0.0010
Epoch 3/60
253/253 ━━━━━━━━━━━━━━━━━━━━ 41s 120ms/step - loss: 0.0293 - mae: 0.0945 - mse: 0.0170 - val_loss: 0.0339 - val_mae: 0.1351 - val_mse: 0.0354 - learning_rate: 0.0010
Epoch 4/60
253/253 ━━━━━━━━━━━━━━━━━━━━ 42s 124ms/step - loss: 0.0213 - mae: 0.0829 - mse: 0.0123 - val_loss: 0.0273 - val_mae: 0.1326 - val_mse: 0.0300 - learning_rate: 0.0010
Epoch 5/60
253/253 ━━━━━━━━━━━━━━━━━━━━ 33s 128ms/step - loss: 0.0185 - mae: 0.0824 - mse: 0.0136 - val_loss: 0.0214 - val_mae: 0.1021 - val_mse: 0.0232 - learning_rate: 0.0010
Epoch 6/60
253/253 ━━━━━━━━━━━━━━━━━━━━ 39s